# 模型测试
使用微调好的model.bin进行测试
 - 文件：

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

tokenizer = AutoTokenizer.from_pretrained("Langboat/mengzi-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("Langboat/mengzi-t5-base")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [2]:
device = "cuda" if torch.cuda.is_available else "cpu"

model.load_state_dict(
    torch.load(
    "./model/epoch_10_valid_bleu_0.26_model_weights.bin", 
    map_location=torch.device(device)
    )
)

<All keys matched successfully>

## 单次问答

In [3]:
# messgge format: context: "" question: ""

qa = """context:长城，又称万里长城，是中国古代的军事防御工事，主要用于抵御北方游牧民族的侵袭。
        其修筑历史可上溯到西周时期，著名的典故"烽火戏诸侯”就源于此。
        question: 长城最初修建的主要目的是什么？"""

inputs = tokenizer(qa, return_tensors="pt")
outputs = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    top_p=0.9,
    top_k=20,
    temperature=1.3,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

抵御北方游牧民族的侵袭


## 多输入问答

In [11]:
# messgge format: context: "" question: ""

qas =[{"context": "Python是一种高级、通用、解释型的编程语言。它由吉多·范罗苏姆创造，并于1991年首次发布。其设计哲学强调代码的可读性。", "question": "Python语言的创始人是谁？"}
    , 
    {"context": "量子计算是一种遵循量子力学规律调控量子信息单元进行计算的新型计算模式。与传统计算机使用比特（0或1）不同，量子计算机使用量子比特，它可以同时处于0和1的叠加状态。", "question": "量子计算机的基本信息单位是什么？"}]


for i, qs in enumerate(qas):
    text = f"context: {qs['context']} qustion: {qs['question']}"
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_length=256,
        do_sample=True,
        top_p=0.9,
        top_k=40,
        temperature=1.2,
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

吉多·范罗苏姆
量子比特
